# 🚀 LifeMate 2026: Chạy Planner & Safe Hash Guard trên Google Colab / VS Code

Notebook này hỗ trợ cả 2 chế độ:
1. **Chạy trực tiếp (Interactive Foreground) với Auto-Resume:** Nhìn thấy từng mẫu chạy trên màn hình. Nếu mất mạng hoặc laptop ngủ, chạy lại sẽ tự động tiếp tục (resume) từ mẫu dở dang mà không mất công chạy lại từ đầu.
2. **Chạy ngầm (Background Runner - GẬP MÁY ĐI NGỦ ĐƯỢC):** Chạy độc lập trên GPU cloud của Colab bằng `nohup`. Bạn có thể gập máy tính, tắt VS Code, GPU trên mây vẫn chạy liên tục cho đến khi xong đủ 300 mẫu.

In [ ]:
# 1. Kiểm tra GPU và VRAM khả dụng
!nvidia-smi
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device: {device_name}')
    print(f'Total VRAM: {vram_gb:.2f} GB')

In [ ]:
# 2. Cài đặt các thư viện cần thiết
!pip install -q -U vllm transformers peft accelerate datasets chunkformer

In [ ]:
# 3. Clone hoặc cập nhật repository ZAIC_2026 trên nhánh planner/vuong
import os
from pathlib import Path

if not os.path.exists('ZAIC_2026'):
    !git clone https://github.com/Bamboohoccode/ZAIC_2026.git
    %cd ZAIC_2026
else:
    %cd /content/ZAIC_2026
    !git fetch origin

!git checkout planner/vuong
!git pull origin planner/vuong
print('\nCommit mới nhất trên nhánh planner/vuong:')
!git log -n 1 --oneline

In [ ]:
# 4. Thiết lập đường dẫn dữ liệu data/ (Dùng luôn nếu đã có, KHÔNG BAO GIỜ giải nén lại)
import os, glob
from pathlib import Path

# 1. NẾU ĐÃ CÓ DATA RỒI THÌ DÙNG LUÔN TRONG 0.001 GIÂY, BỎ QUA HOÀN TOÀN GIẢI NÉN!
if os.path.exists('data/train/train.jsonl'):
    print('✅ Dữ liệu data/train đã có sẵn đầy đủ, dùng luôn không cần giải nén lại!')
    !ls -lh data/train | head -n 5
else:
    # Dọn dẹp nếu có symlink cũ bị lỗi
    if os.path.islink('data/train') or (os.path.exists('data/train') and not os.listdir('data/train')):
        os.system('rm -rf data/train')
    os.makedirs('data', exist_ok=True)

    # 2. Kiểm tra xem có thư mục data/train nào đã giải nén sẵn không (ưu tiên symlink trong 0.01s)
    candidates = [
        Path('/content/data/train'),
        Path('/content/train'),
        Path('/content/drive/MyDrive/data/train'),
        Path('/content/drive/MyDrive/ZAIC_2026/data/train'),
        Path('../data/train'),
    ]
    found = None
    for c in candidates:
        if (c / 'train.jsonl').exists():
            found = c.resolve()
            break

    if found and str(found) != str(Path('data/train').resolve()):
        print(f'✅ Đã tìm thấy thư mục giải nén sẵn tại: {found}. Tạo liên kết ngay...')
        !ln -s "{found}" data/train
    else:
        # 3. CHỈ KHI CHƯA CÓ BẤT KỲ ĐÂU MỚI GIẢI NÉN DATA.ZIP
        zip_paths = ['/content/data.zip', 'data.zip', '/content/drive/MyDrive/data.zip']
        for z in zip_paths:
            if os.path.exists(z):
                print(f'📦 Đang giải nén lần đầu từ {z}...')
                !unzip -q -o "{z}" -d .
                break

    if os.path.exists('data/train/train.jsonl'):
        print('✅ Thiết lập data/train thành công:')
        !ls -lh data/train | head -n 5
    else:
        print('⚠️ Chưa tìm thấy train.jsonl! Hãy kiểm tra lại đường dẫn.')


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

### 🎯 LỰA CHỌN CÁCH CHẠY PIPELINE:
- **Cách 5A (Khuyên dùng khi đi ngủ):** Chạy ngầm trong background với `nohup`. Bạn có thể **gập máy tính lại**, GPU Colab vẫn chạy tiếp đến khi xong!
- **Cách 5B (Khi đang ngồi máy):** Chạy trực tiếp trên màn hình có kèm cờ `--resume` (nếu lỡ tắt máy, chạy lại sẽ tự làm tiếp từ mẫu vừa dừng).

In [ ]:
# 5A. [KHUYÊN DÙNG] CHẠY NGẦM BACKGROUND (GẬP MÁY / SLEEP LAPTOP THOẢI MÁI)
# Dọn dẹp tiến trình cũ để giải phóng 100% VRAM GPU
!pkill -9 -f 'lifemate.pipeline' || true
import os, torch, time
time.sleep(2)
os.makedirs('runs', exist_ok=True)

vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 16
gpu_mem = 0.88 if vram < 20 else 0.90
print(f'GPU VRAM: {vram:.1f} GB -> Cấu hình --gpu-mem {gpu_mem}')

# Chạy ngầm bằng nohup: không bị ngắt khi gập máy tính
!nohup python -m lifemate.pipeline \
    --root data \
    --split train \
    --text \
    --llm Qwen/Qwen3-4B-Instruct-2507 \
    --mapper none \
    --gpu-mem {gpu_mem} \
    --max-ctx 8192 \
    --verify-rounds 0 \
    --prompt prompts/gepa/vuong/planner_v4.txt \
    --ids eval/splits/dev_ids.txt \
    --resume \
    --out runs/dev_v4_guarded.jsonl > runs/dev_v4_guarded.nohup.log 2>&1 &

print('🚀 Tiến trình đã được đưa vào chạy ngầm trên GPU Cloud!')
print('Bạn có thể gập máy tính lại hoặc theo dõi tiến độ ở Cell 5C bên dưới.')

In [ ]:
# 5B. (Tuỳ chọn) CHẠY TRỰC TIẾP TRÊN MÀN HÌNH (CÓ TỰ ĐỘNG RESUME TIẾP TỤC)
# import os, torch
# os.makedirs('runs', exist_ok=True)
# vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 16
# gpu_mem = 0.88 if vram < 20 else 0.90
#
# !python -m lifemate.pipeline \
#     --root data \
#     --split train \
#     --text \
#     --llm Qwen/Qwen3-4B-Instruct-2507 \
#     --mapper none \
#     --gpu-mem {gpu_mem} \
#     --max-ctx 8192 \
#     --verify-rounds 0 \
#     --prompt prompts/gepa/vuong/planner_v4.txt \
#     --ids eval/splits/dev_ids.txt \
#     --resume \
#     --out runs/dev_v4_guarded.jsonl

In [ ]:
# 5C. THEO DÕI TIẾN ĐỘ THỜI GIAN THỰC (Chạy cell này bất cứ lúc nào để kiểm tra)
import time, os
from IPython.display import clear_output

out_path = 'runs/dev_v4_guarded.jsonl'
log_path = 'runs/dev_v4_guarded.jsonl.log'

for _ in range(60):  # cập nhật mỗi 5s trong 5 phút
    clear_output(wait=True)
    done_count = 0
    if os.path.exists(out_path):
        with open(out_path, 'r', encoding='utf-8') as f:
            done_count = sum(1 for line in f if line.strip())
    
    print(f'📊 TIẾN ĐỘ: Đã xong {done_count}/300 mẫu dev ({done_count/300*100:.1f}%)')
    print('=' * 60)
    if os.path.exists(log_path):
        !tail -n 12 runs/dev_v4_guarded.jsonl.log
    
    is_running = os.popen("pgrep -f 'lifemate.pipeline'").read().strip()
    if is_running:
        print(f'\n⚡ Tiến trình GPU đang chạy (PID: {is_running})...')
    else:
        print('\n⏹️ Tiến trình đã hoàn thành xong toàn bộ hoặc đã dừng.')
        break
    time.sleep(5)

In [ ]:
# 6. Chấm điểm và phân tích lỗi chi tiết với eval/scorer.py
!python eval/scorer.py --sub runs/dev_v4_guarded.jsonl

In [ ]:
# 7. Tự động sao lưu file kết quả và run log vào Google Drive
import os, shutil
from pathlib import Path

if os.path.exists('/content/drive/MyDrive'):
    backup_dir = Path('/content/drive/MyDrive/ZAIC_2026_runs')
    backup_dir.mkdir(parents=True, exist_ok=True)
    for fname in ['dev_v4_guarded.jsonl', 'dev_v4_guarded.jsonl.log', 'dev_v4_guarded.jsonl.trace.jsonl']:
        fpath = Path('runs') / fname
        if fpath.exists():
            shutil.copy(fpath, backup_dir / fname)
            print(f'✅ Đã lưu: {fname} -> {backup_dir}')
    print(f'Toàn bộ kết quả đã được sao lưu an toàn tại Google Drive: {backup_dir}')
else:
    print('File kết quả hiện nằm trực tiếp tại: runs/dev_v4_guarded.jsonl')

In [ ]:
!pkill -9 -f 'lifemate.pipeline'

In [ ]:
!nvidia-smi